# Build `known_aircraft.csv` from current-operator aircraft lists

This notebook creates a `known_aircraft.csv` file compatible with the `--known-aircraft` option used by `ingest-graph`. It is designed for Wikipedia articles that list currently operational military aircraft, such as `https://en.wikipedia.org/wiki/List_of_active_Russian_military_aircraft`.

The primary path is deterministic HTML table extraction using only the Python standard library. If a page cannot be parsed into a useful table, or if you want LLM-assisted cleanup, the optional Ollama cell asks a local Ollama model to return strict JSON records.


In [ ]:
from __future__ import annotations

import ast
import csv
import html.parser
import json
import re
import urllib.parse
import urllib.request
from pathlib import Path
from typing import Iterable


## Configuration

Add one or more article URLs. The output file uses the first column as the aircraft name, which is the only field required by `read_known_aircraft`; additional columns are retained for auditability.


In [ ]:
ARTICLE_URLS = [
    "https://en.wikipedia.org/wiki/List_of_active_Russian_military_aircraft",
    'https://en.wikipedia.org/wiki/Ukrainian_Air_Force#Equipment',
    'https://en.wikipedia.org/wiki/List_of_active_United_Kingdom_military_aircraft',
    'https://en.wikipedia.org/wiki/List_of_active_Indian_military_aircraft',
    'https://en.wikipedia.org/wiki/Azerbaijani_Air_Forces#Aircraft',
    # Add more current-inventory list pages here, for example:
    # "https://en.wikipedia.org/wiki/List_of_active_United_States_military_aircraft",
]

OUTPUT_CSV = Path("../known_aircraft.csv")

# Optional LLM cleanup/extraction fallback. Set USE_OLLAMA = True when table parsing is insufficient.
USE_OLLAMA = False
OLLAMA_URL = "http://localhost:11434"
OLLAMA_MODEL = "qwen3.5:9b"


## Deterministic table extraction

The parser keeps rows from tables that look like active aircraft inventories. It identifies an aircraft/model/name column, ignores totals and section headers, and preserves role, origin, operator, quantity, source URL, and source table for review.


In [ ]:
class WikipediaTableParser(html.parser.HTMLParser):
    """Extract table rows from Wikipedia-ish HTML without third-party dependencies."""

    def __init__(self) -> None:
        super().__init__()
        self.tables: list[list[list[str]]] = []
        self._current_table: list[list[str]] | None = None
        self._current_row: list[str] | None = None
        self._current_cell_parts: list[str] | None = None
        self._in_table = 0
        self._skip_depth = 0

    def handle_starttag(self, tag: str, attrs: list[tuple[str, str | None]]) -> None:
        if tag in {"script", "style", "sup"}:
            self._skip_depth += 1
            return
        if tag == "table":
            self._in_table += 1
            if self._in_table == 1:
                self._current_table = []
        elif tag == "tr" and self._in_table == 1:
            self._current_row = []
        elif tag in {"td", "th"} and self._current_row is not None:
            self._current_cell_parts = []
        elif tag == "br" and self._current_cell_parts is not None:
            self._current_cell_parts.append("; ")

    def handle_endtag(self, tag: str) -> None:
        if tag in {"script", "style", "sup"} and self._skip_depth:
            self._skip_depth -= 1
            return
        if tag in {"td", "th"} and self._current_cell_parts is not None and self._current_row is not None:
            self._current_row.append(clean_cell("".join(self._current_cell_parts)))
            self._current_cell_parts = None
        elif tag == "tr" and self._current_row is not None:
            if any(cell for cell in self._current_row):
                self._current_table.append(self._current_row)  # type: ignore[union-attr]
            self._current_row = None
        elif tag == "table" and self._in_table:
            if self._in_table == 1 and self._current_table:
                self.tables.append(self._current_table)
                self._current_table = None
            self._in_table -= 1

    def handle_data(self, data: str) -> None:
        if self._skip_depth:
            return
        if self._current_cell_parts is not None:
            self._current_cell_parts.append(data)


AIRCRAFT_COLUMN_HINTS = ("aircraft", "type", "model", "name", "platform")
SKIP_NAME_PATTERNS = re.compile(r"^(total|notes?|references?|see also|aircraft|type|model)$", re.I)
BRACKETED_REF_PATTERN = re.compile(r"\[[^\]]+\]")
PAREN_TRAILER_PATTERN = re.compile(r"\s*\([^)]*(?:planned|ordered|option|former|retired|stored)[^)]*\)", re.I)


def clean_cell(value: object) -> str:
    if value is None:
        return ""
    text = str(value)
    text = BRACKETED_REF_PATTERN.sub("", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def clean_aircraft_name(value: object) -> str:
    name = clean_cell(value)
    name = PAREN_TRAILER_PATTERN.sub("", name)
    name = re.sub(r"\s*/\s*", " / ", name).strip(" -–—;.,")
    return name


def fetch_html(url: str) -> str:
    request = urllib.request.Request(url, headers={"User-Agent": "CMO-Sensor-Fusion/0.1 known-aircraft-notebook"})
    with urllib.request.urlopen(request, timeout=30) as response:
        return response.read().decode("utf-8", errors="replace")


def extract_tables(html: str) -> list[list[list[str]]]:
    parser = WikipediaTableParser()
    parser.feed(html)
    return parser.tables


def find_header_index(rows: list[list[str]]) -> int | None:
    for index, row in enumerate(rows[:5]):
        lowered = [cell.lower() for cell in row]
        if any(any(hint in cell for hint in AIRCRAFT_COLUMN_HINTS) for cell in lowered):
            return index
    return 0 if rows else None


def find_column(headers: list[str], *hints: str) -> int | None:
    lowered = [header.lower() for header in headers]
    for hint in hints:
        for index, header in enumerate(lowered):
            if hint in header:
                return index
    return None


def cell_at(row: list[str], index: int | None) -> str:
    if index is None or index >= len(row):
        return ""
    return clean_cell(row[index])


def table_records_from_url(url: str) -> list[dict[str, str]]:
    records: list[dict[str, str]] = []
    for table_index, rows in enumerate(extract_tables(fetch_html(url)), start=1):
        header_index = find_header_index(rows)
        if header_index is None:
            continue
        headers = [clean_cell(cell).lower() for cell in rows[header_index]]
        aircraft_index = find_column(headers, *AIRCRAFT_COLUMN_HINTS)
        if aircraft_index is None:
            aircraft_index = 0
        role_index = find_column(headers, "role", "type")
        origin_index = find_column(headers, "origin", "country")
        operator_index = find_column(headers, "operator", "service", "branch")
        quantity_index = find_column(headers, "quantity", "in service", "active", "number")
        for row in rows[header_index + 1 :]:
            aircraft = clean_aircraft_name(cell_at(row, aircraft_index))
            if not aircraft or SKIP_NAME_PATTERNS.match(aircraft):
                continue
            if len(aircraft) < 2 or aircraft.lower() in {"nan", "none"}:
                continue
            records.append({
                "aircraft": aircraft,
                "role": cell_at(row, role_index),
                "origin": cell_at(row, origin_index),
                "operator": cell_at(row, operator_index),
                "quantity": cell_at(row, quantity_index),
                "source_url": url,
                "source_table": str(table_index),
                "extraction_method": "stdlib_html_table",
            })
    return records


## Optional Ollama extraction fallback

Use this only when table parsing misses a page or a page uses prose instead of tables. The prompt requires current operational aircraft only and returns JSON records with the same `aircraft` field expected by the CSV writer.


In [ ]:
def html_to_text(html: str) -> str:
    text = re.sub(r"<script.*?</script>|<style.*?</style>", " ", html, flags=re.I | re.S)
    text = re.sub(r"<[^>]+>", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def html_to_relevant_text(url: str, html: str, max_chars: int = 24000) -> str:
    """Return article text focused on the URL fragment when one is present.

    Wikipedia articles such as Ukrainian_Air_Force#Equipment contain many tables outside
    the current aircraft inventory. When a fragment is supplied, trim the HTML to that
    section (through the next heading at the same or higher level) before stripping tags
    so the LLM sees the aircraft inventory without unrelated history, commanders,
    air-defense, radar, or armament tables from the rest of the article.
    """

    fragment = urllib.parse.urlparse(url).fragment
    if fragment:
        fragment_pattern = re.escape(urllib.parse.unquote(fragment)).replace(r"\ ", r"[ _]")
        anchor_match = re.search(
            rf'<(?:span|h[1-6])[^>]+id=["\']{fragment_pattern}["\'][^>]*>',
            html,
            flags=re.I,
        )
        if anchor_match:
            heading_start = html.rfind("<h", 0, anchor_match.start())
            if heading_start == -1:
                heading_start = anchor_match.start()
            heading_match = re.match(r"<h([1-6])\b", html[heading_start:], flags=re.I)
            heading_level = int(heading_match.group(1)) if heading_match else 2
            next_heading = re.search(rf"<h[1-{heading_level}]\b", html[anchor_match.end() :], flags=re.I)
            section_end = anchor_match.end() + next_heading.start() if next_heading else len(html)
            section_text = html_to_text(html[heading_start:section_end])
            if section_text:
                return section_text[:max_chars]

    return html_to_text(html)[:max_chars]


def extract_balanced_json_values(text: str) -> list[str]:
    """Return balanced JSON object/array candidates, ignoring braces inside strings."""

    candidates: list[str] = []
    open_to_close = {"{": "}", "[": "]"}
    for start, start_char in enumerate(text):
        if start_char not in open_to_close:
            continue
        stack = [open_to_close[start_char]]
        in_string = False
        escape = False
        for index, char in enumerate(text[start + 1 :], start=start + 1):
            if escape:
                escape = False
                continue
            if char == "\\" and in_string:
                escape = True
                continue
            if char == '"':
                in_string = not in_string
                continue
            if in_string:
                continue
            if char in open_to_close:
                stack.append(open_to_close[char])
            elif stack and char == stack[-1]:
                stack.pop()
                if not stack:
                    candidates.append(text[start : index + 1])
                    break
    return candidates


def parse_ollama_json_response(raw: str) -> dict:
    """Parse records from an Ollama response, returning an empty object on failure.

    Some local models occasionally ignore the JSON-schema response contract and emit
    Python-style repr fragments such as ``{'records': [...]}`` or one or more bare
    lists. Try strict JSON first, then safe literal parsing of balanced object/array
    fragments so a malformed leading fragment does not hide a later usable list.
    """

    candidates = [raw.strip()]
    candidates.extend(
        match.group(1).strip()
        for match in re.finditer(r"```(?:json|python)?\s*(.*?)```", raw, flags=re.I | re.S)
    )
    candidates.extend(extract_balanced_json_values(raw))

    seen: set[str] = set()
    empty_records_payload: dict | None = None
    last_error: Exception | None = None
    for candidate in candidates:
        if not candidate or candidate in seen:
            continue
        seen.add(candidate)
        for parser in (json.loads, ast.literal_eval):
            try:
                parsed = parser(candidate)
                if isinstance(parsed, str):
                    parsed = json.loads(parsed)
            except (json.JSONDecodeError, SyntaxError, ValueError) as exc:
                last_error = exc
                continue
            if isinstance(parsed, dict):
                records = parsed.get("records")
                if isinstance(records, list):
                    if records:
                        return parsed
                    empty_records_payload = parsed
                continue
            if isinstance(parsed, list):
                payload = {"records": parsed}
                if parsed:
                    return payload
                empty_records_payload = payload

    if empty_records_payload is not None:
        return empty_records_payload

    if last_error:
        print(f"Warning: Could not parse Ollama JSON response ({last_error}); skipping response preview: {raw[:200]!r}")
    else:
        print(f"Warning: Ollama returned non-JSON text; skipping response: {raw[:200]!r}")
    return {}


def ollama_extract_current_aircraft(url: str, model: str = OLLAMA_MODEL, ollama_url: str = OLLAMA_URL) -> list[dict[str, str]]:
    page_text = html_to_relevant_text(url, fetch_html(url))
    prompt = f"""You are extracting a machine-readable aircraft inventory from article text.

OUTPUT FORMAT REQUIREMENTS (must follow exactly):
- Return only valid JSON. Do not include Markdown fences, comments, explanations, or leading/trailing prose.
- Return exactly one JSON object with this shape:
  {{"records":[{{"aircraft":"string","role":"string","origin":"string","operator":"string","quantity":"string","evidence":"string"}}]}}
- The top-level value must be an object, not an array.
- The only top-level key must be "records".
- "records" must be an array; return {{"records":[]}} if no current aircraft are found.
- Every record must contain all six string fields: aircraft, role, origin, operator, quantity, evidence.
- Use double quotes for all JSON strings and property names. Escape embedded quotes. Do not use trailing commas.
- Keep evidence short and quote or paraphrase the source row/phrase that proves current service.

EXTRACTION RULES:
- Only include aircraft currently in active/current service in the article's current inventory or equipment section.
- For complex force pages, use only tables or prose that describe current aircraft/equipment inventories.
- Ignore unrelated tables such as commanders, bases, structure, losses, orders, accidents, wars, air-defense systems, armament, radars, missiles, vehicles, ships, sensors, and references.
- Include fixed-wing aircraft, helicopters, trainers, transports, tankers, AEW/C2 aircraft, reconnaissance aircraft, and UAV/UCAV aircraft when they are current operational inventory.
- Exclude retired, former, historical, captured-only, lost/destroyed, ordered-only, planned-only, on-option, museum, and future aircraft.
- Do not include radars, missiles, weapons, ships, ground vehicles, air-defense systems, or sensors as aircraft.
- If a row names a non-aircraft item, omit it even if it appears in an Equipment section.

Article URL: {url}
Article text:
{page_text}
"""
    payload = json.dumps({
        "model": model,
        "prompt": prompt,
        "stream": False,
        "format": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "records": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "additionalProperties": False,
                        "properties": {
                            "aircraft": {"type": "string"},
                            "role": {"type": "string"},
                            "origin": {"type": "string"},
                            "operator": {"type": "string"},
                            "quantity": {"type": "string"},
                            "evidence": {"type": "string"},
                        },
                        "required": ["aircraft", "role", "origin", "operator", "quantity", "evidence"],
                    },
                },
            },
            "required": ["records"],
        },
        "options": {"temperature": 0, "num_predict": 8192},
        "think": False,
    }).encode("utf-8")
    request = urllib.request.Request(
        f"{ollama_url.rstrip('/')}/api/generate",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=180) as response:
        data = json.loads(response.read().decode("utf-8"))
    raw = data.get("response") or data.get("thinking") or "{}"
    parsed = parse_ollama_json_response(raw)
    records = parsed.get("records", [])
    cleaned: list[dict[str, str]] = []
    for record in records:
        if not isinstance(record, dict):
            continue
        aircraft = clean_aircraft_name(record.get("aircraft", ""))
        if not aircraft:
            continue
        cleaned.append({
            "aircraft": aircraft,
            "role": clean_cell(record.get("role", "")),
            "origin": clean_cell(record.get("origin", "")),
            "operator": clean_cell(record.get("operator", "")),
            "quantity": clean_cell(record.get("quantity", "")),
            "source_url": url,
            "source_table": "",
            "extraction_method": f"ollama:{model}",
        })
    return cleaned


## Build and review records

Run the next cell to fetch each article, extract aircraft rows, de-duplicate by aircraft/operator/source, and display a sample before writing CSV.


In [ ]:
all_records: list[dict[str, str]] = []
for url in ARTICLE_URLS:
    parsed_records = table_records_from_url(url)
    if USE_OLLAMA or not parsed_records:
        parsed_records.extend(ollama_extract_current_aircraft(url))
    all_records.extend(parsed_records)

# De-duplicate while preserving useful audit columns.
seen: set[tuple[str, str, str]] = set()
deduped: list[dict[str, str]] = []
for record in all_records:
    key = (record["aircraft"].casefold(), record.get("operator", "").casefold(), record.get("source_url", ""))
    if key in seen:
        continue
    seen.add(key)
    deduped.append(record)

known_aircraft = sorted(deduped, key=lambda item: (item["aircraft"].casefold(), item.get("operator", "")))
print(f"Extracted {len(known_aircraft)} aircraft records from {len(ARTICLE_URLS)} article(s).")
for record in known_aircraft[:25]:
    print(record)


## Write `known_aircraft.csv`

The first column is `aircraft`, so the file is directly compatible with `read_known_aircraft` and `ingest-graph --known-aircraft`.


In [ ]:
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
fieldnames = ["aircraft", "role", "origin", "operator", "quantity", "source_url", "source_table", "extraction_method"]
with OUTPUT_CSV.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows({field: record.get(field, "") for field in fieldnames} for record in known_aircraft)

print(f"Wrote {len(known_aircraft)} rows to {OUTPUT_CSV.resolve()}")


## Use the output with graph ingestion

```bash
python -m combat_id_calibration ingest-graph \
  --wikipedia https://en.wikipedia.org/wiki/Zhuk_radar \
  --neo4j-password "$NEO4J_PASSWORD" \
  --known-aircraft known_aircraft.csv
```
